In [1]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# Latihan 1: CNN dari Scratch untuk CIFAR-10
def praktikum_cnn_from_scratch():
    print("LATIHAN 1: CNN DARI AWAL UNTUK CIFAR-10")
    print("=" * 50)
    
    # Load CIFAR-10 dataset
    (X_train, y_train), (X_test, y_test) = keras.datasets.cifar10.load_data()
    
    print(f"Training data shape: {X_train.shape}")
    print(f"Test data shape: {X_test.shape}")
    
    # Normalize pixel values to [0, 1]
    X_train = X_train.astype('float32') / 255.0
    X_test = X_test.astype('float32') / 255.0
    
    # One-hot encode labels
    num_classes = 10
    y_train = keras.utils.to_categorical(y_train, num_classes)
    y_test = keras.utils.to_categorical(y_test, num_classes)
    
    # Visualize sample images
    class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer', 
                   'dog', 'frog', 'horse', 'ship', 'truck']
    
    plt.figure(figsize=(10, 10))
    for i in range(25):
        plt.subplot(5, 5, i + 1)
        plt.imshow(X_train[i])
        plt.title(class_names[np.argmax(y_train[i])])
        plt.axis('off')
    plt.suptitle("Sample Images from CIFAR-10", fontsize=16)
    plt.tight_layout()
    plt.show()
    
    # Build CNN from scratch
    def build_cnn_model(input_shape=(32, 32, 3), num_classes=10):
        model = keras.Sequential([
            # Convolutional Block 1
            layers.Conv2D(32, (3, 3), padding='same', activation='relu', 
                         input_shape=input_shape, name='conv1'),
            layers.BatchNormalization(name='bn1'),
            layers.Conv2D(32, (3, 3), padding='same', activation='relu', name='conv2'),
            layers.BatchNormalization(name='bn2'),
            layers.MaxPooling2D((2, 2), name='pool1'),
            layers.Dropout(0.25, name='dropout1'),
            
            # Convolutional Block 2
            layers.Conv2D(64, (3, 3), padding='same', activation='relu', name='conv3'),
            layers.BatchNormalization(name='bn3'),
            layers.Conv2D(64, (3, 3), padding='same', activation='relu', name='conv4'),
            layers.BatchNormalization(name='bn4'),
            layers.MaxPooling2D((2, 2), name='pool2'),
            layers.Dropout(0.25, name='dropout2'),
            
            # Convolutional Block 3
            layers.Conv2D(128, (3, 3), padding='same', activation='relu', name='conv5'),
            layers.BatchNormalization(name='bn5'),
            layers.Conv2D(128, (3, 3), padding='same', activation='relu', name='conv6'),
            layers.BatchNormalization(name='bn6'),
            layers.MaxPooling2D((2, 2), name='pool3'),
            layers.Dropout(0.25, name='dropout3'),
            
            # Fully Connected Layers
            layers.Flatten(name='flatten'),
            layers.Dense(256, activation='relu', name='fc1'),
            layers.BatchNormalization(name='bn7'),
            layers.Dropout(0.5, name='dropout4'),
            layers.Dense(num_classes, activation='softmax', name='output')
        ])
        
        return model
    
    # Create model
    model = build_cnn_model()
    
    # Display model architecture
    print("\nCNN MODEL ARCHITECTURE:")
    print("=" * 40)
    model.summary()
    
    # Visualize model architecture
    keras.utils.plot_model(model, show_shapes=True, show_layer_names=True)
    
    # Compile model
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    # Callbacks
    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor='val_accuracy',
            patience=10,
            restore_best_weights=True
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=5,
            min_lr=1e-6
        )
    ]
    
    # Train model
    print("\nTRAINING CNN FROM SCRATCH...")
    history = model.fit(
        X_train, y_train,
        batch_size=64,
        epochs=50,
        validation_split=0.2,
        callbacks=callbacks,
        verbose=1
    )
    
    # Evaluate model
    test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
    print(f"\nTest Accuracy: {test_accuracy:.4f}")
    print(f"Test Loss: {test_loss:.4f}")
    
    # Plot training history
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    # Accuracy plot
    ax1.plot(history.history['accuracy'], label='Training Accuracy')
    ax1.plot(history.history['val_accuracy'], label='Validation Accuracy')
    ax1.set_title('Model Accuracy')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Accuracy')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Loss plot
    ax2.plot(history.history['loss'], label='Training Loss')
    ax2.plot(history.history['val_loss'], label='Validation Loss')
    ax2.set_title('Model Loss')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Visualize predictions
    y_pred = model.predict(X_test[:25])
    y_pred_classes = np.argmax(y_pred, axis=1)
    y_true_classes = np.argmax(y_test[:25], axis=1)
    
    plt.figure(figsize=(10, 10))
    for i in range(25):
        plt.subplot(5, 5, i + 1)
        plt.imshow(X_test[i])
        
        # Color code predictions
        color = 'green' if y_pred_classes[i] == y_true_classes[i] else 'red'
        plt.title(f"True: {class_names[y_true_classes[i]]}\nPred: {class_names[y_pred_classes[i]]}", 
                 color=color, fontsize=8)
        plt.axis('off')
    
    plt.suptitle("Sample Predictions (Green=Correct, Red=Wrong)", fontsize=14)
    plt.tight_layout()
    plt.show()
    
    # Visualize feature maps from first convolutional layer
    print("\nVISUALIZING FEATURE MAPS...")
    layer_outputs = [layer.output for layer in model.layers[:8]]  # First 8 layers
    activation_model = keras.Model(inputs=model.input, outputs=layer_outputs)
    
    # Get activations for a sample image
    sample_image = X_test[0:1]
    activations = activation_model.predict(sample_image)
    
    # Visualize feature maps
    layer_names = [layer.name for layer in model.layers[:8]]
    
    for layer_name, layer_activation in zip(layer_names, activations):
        if 'conv' in layer_name:
            n_features = layer_activation.shape[-1]
            size = layer_activation.shape[1]
            
            n_cols = 8
            n_rows = n_features // n_cols
            
            fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols*2, n_rows*2))
            fig.suptitle(f'Feature Maps: {layer_name}', fontsize=16)
            
            for i in range(n_features):
                ax = axes[i // n_cols, i % n_cols]
                ax.imshow(layer_activation[0, :, :, i], cmap='viridis')
                ax.axis('off')
            
            plt.tight_layout()
            plt.show()
            break  # Show only first conv layer
    
    return model, history, test_accuracy

# Jalankan latihan 1
cnn_model, training_history, cnn_accuracy = praktikum_cnn_from_scratch()

ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
# Latihan 2: Transfer Learning dengan Data Augmentation
def praktikum_transfer_learning():
    print("\nLATIHAN 2: TRANSFER LEARNING DENGAN DATA AUGMENTATION")
    print("=" * 60)
    
    # Load smaller dataset untuk demonstrasi transfer learning
    # Use Cats vs Dogs dataset subset
    import tensorflow_datasets as tfds
    
    # Create synthetic dataset untuk demonstrasi
    def create_synthetic_dataset(num_samples=1000, img_size=150):
        np.random.seed(42)
        
        # Class 0: Circles (simulating cats)
        X_circles = np.zeros((num_samples, img_size, img_size, 3))
        y_circles = np.zeros(num_samples)
        
        # Class 1: Triangles (simulating dogs)
        X_triangles = np.zeros((num_samples, img_size, img_size, 3))
        y_triangles = np.ones(num_samples)
        
        for i in range(num_samples):
            # Create circle
            img = np.ones((img_size, img_size, 3)) * 0.8  # Light background
            center_x = np.random.randint(30, img_size-30)
            center_y = np.random.randint(30, img_size-30)
            radius = np.random.randint(20, 40)
            color = np.random.random(3) * 0.5 + 0.2  # Random dark color
            cv2.circle(img, (center_x, center_y), radius, color, -1)
            X_circles[i] = img
            y_circles[i] = 0
            
            # Create triangle
            img = np.ones((img_size, img_size, 3)) * 0.8
            pt1 = (np.random.randint(20, img_size-20), np.random.randint(20, img_size-60))
            pt2 = (pt1[0] - 30, pt1[1] + 60)
            pt3 = (pt1[0] + 30, pt1[1] + 60)
            color = np.random.random(3) * 0.5 + 0.2
            cv2.fillPoly(img, [np.array([pt1, pt2, pt3])], color)
            X_triangles[i] = img
            y_triangles[i] = 1
        
        X = np.vstack([X_circles, X_triangles])
        y = np.hstack([y_circles, y_triangles])
        
        return X, y
    
    # Create synthetic dataset
    X, y = create_synthetic_dataset(500, 150)
    
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    
    X_train, X_val, y_train, y_val = train_test_split(
        X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
    )
    
    print(f"Training samples: {len(X_train)}")
    print(f"Validation samples: {len(X_val)}")
    print(f"Test samples: {len(X_test)}")
    
    # Visualize dataset
    plt.figure(figsize=(10, 5))
    for i in range(10):
        plt.subplot(2, 5, i + 1)
        plt.imshow(X_train[i])
        plt.title(f"Class: {'Cat' if y_train[i] == 0 else 'Dog'}")
        plt.axis('off')
    plt.suptitle("Sample Images from Synthetic Dataset", fontsize=14)
    plt.tight_layout()
    plt.show()
    
    # Data Augmentation Layer
    data_augmentation = keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.1),
        layers.RandomZoom(0.1),
        layers.RandomContrast(0.1),
        layers.RandomBrightness(0.1),
    ], name='data_augmentation')
    
    # Visualize augmented images
    plt.figure(figsize=(15, 5))
    for i in range(5):
        plt.subplot(2, 5, i + 1)
        plt.imshow(X_train[i])
        plt.title(f"Original\nClass: {'Cat' if y_train[i] == 0 else 'Dog'}")
        plt.axis('off')
        
        plt.subplot(2, 5, i + 6)
        augmented = data_augmentation(tf.expand_dims(X_train[i], 0))
        plt.imshow(augmented[0].numpy())
        plt.title("Augmented")
        plt.axis('off')
    
    plt.suptitle("Data Augmentation Examples", fontsize=14)
    plt.tight_layout()
    plt.show()
    
    # Transfer Learning dengan VGG16
    print("\nIMPLEMENTING TRANSFER LEARNING WITH VGG16...")
    
    # Load pre-trained VGG16
    vgg_base = keras.applications.VGG16(
        weights='imagenet',
        include_top=False,
        input_shape=(150, 150, 3)
    )
    
    # Freeze convolutional base
    vgg_base.trainable = False
    
    print(f"VGG16 base layers: {len(vgg_base.layers)}")
    print("VGG16 base frozen for feature extraction")
    
    # Build transfer learning model
    def build_transfer_model(base_model, augmentation=None):
        inputs = keras.Input(shape=(150, 150, 3))
        
        if augmentation:
            x = augmentation(inputs)
        else:
            x = inputs
        
        # Preprocess for VGG
        x = keras.applications.vgg16.preprocess_input(x)
        
        # Base model (VGG16)
        x = base_model(x, training=False)
        
        # Add new layers
        x = layers.GlobalAveragePooling2D()(x)
        x = layers.Dense(256, activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.5)(x)
        x = layers.Dense(128, activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.3)(x)
        outputs = layers.Dense(1, activation='sigmoid')(x)
        
        model = keras.Model(inputs, outputs)
        return model
    
    # Build model dengan data augmentation
    model_aug = build_transfer_model(vgg_base, data_augmentation)
    
    # Build model tanpa data augmentation (untuk perbandingan)
    model_no_aug = build_transfer_model(vgg_base)
    
    print("\nTRANSFER LEARNING MODEL ARCHITECTURE:")
    model_aug.summary()
    
    # Compile models
    model_aug.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss='binary_crossentropy',
        metrics=['accuracy', keras.metrics.Precision(), keras.metrics.Recall()]
    )
    
    model_no_aug.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    
    # Callbacks
    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor='val_accuracy',
            patience=10,
            restore_best_weights=True
        ),
        keras.callbacks.ModelCheckpoint(
            'best_transfer_model.h5',
            monitor='val_accuracy',
            save_best_only=True
        )
    ]
    
    # Train both models
    print("\nTRAINING MODELS...")
    print("Model with Data Augmentation:")
    history_aug = model_aug.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=30,
        batch_size=32,
        callbacks=callbacks,
        verbose=1
    )
    
    print("\nModel without Data Augmentation:")
    history_no_aug = model_no_aug.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=30,
        batch_size=32,
        verbose=1
    )
    
    # Evaluate models
    print("\nMODEL EVALUATION:")
    print("-" * 40)
    
    # Model with augmentation
    test_loss_aug, test_acc_aug, test_prec_aug, test_rec_aug = model_aug.evaluate(
        X_test, y_test, verbose=0
    )
    test_f1_aug = 2 * (test_prec_aug * test_rec_aug) / (test_prec_aug + test_rec_aug)
    
    # Model without augmentation
    test_loss_no_aug, test_acc_no_aug = model_no_aug.evaluate(X_test, y_test, verbose=0)
    
    print(f"With Augmentation - Accuracy: {test_acc_aug:.4f}, F1-Score: {test_f1_aug:.4f}")
    print(f"Without Augmentation - Accuracy: {test_acc_no_aug:.4f}")
    
    # Plot comparison
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    
    # Accuracy comparison
    axes[0, 0].plot(history_aug.history['accuracy'], label='Train (Aug)', color='blue')
    axes[0, 0].plot(history_aug.history['val_accuracy'], label='Val (Aug)', linestyle='--', color='blue')
    axes[0, 0].plot(history_no_aug.history['accuracy'], label='Train (No Aug)', color='red')
    axes[0, 0].plot(history_no_aug.history['val_accuracy'], label='Val (No Aug)', linestyle='--', color='red')
    axes[0, 0].set_title('Accuracy Comparison')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Accuracy')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # Loss comparison
    axes[0, 1].plot(history_aug.history['loss'], label='Train (Aug)', color='blue')
    axes[0, 1].plot(history_aug.history['val_loss'], label='Val (Aug)', linestyle='--', color='blue')
    axes[0, 1].plot(history_no_aug.history['loss'], label='Train (No Aug)', color='red')
    axes[0, 1].plot(history_no_aug.history['val_loss'], label='Val (No Aug)', linestyle='--', color='red')
    axes[0, 1].set_title('Loss Comparison')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Loss')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # Bar chart comparison
    metrics = ['Train Acc', 'Val Acc', 'Test Acc']
    aug_scores = [
        history_aug.history['accuracy'][-1],
        history_aug.history['val_accuracy'][-1],
        test_acc_aug
    ]
    no_aug_scores = [
        history_no_aug.history['accuracy'][-1],
        history_no_aug.history['val_accuracy'][-1],
        test_acc_no_aug
    ]
    
    x = np.arange(len(metrics))
    width = 0.35
    
    axes[0, 2].bar(x - width/2, aug_scores, width, label='With Augmentation', color='blue')
    axes[0, 2].bar(x + width/2, no_aug_scores, width, label='Without Augmentation', color='red')
    axes[0, 2].set_title('Final Performance Comparison')
    axes[0, 2].set_xlabel('Metric')
    axes[0, 2].set_ylabel('Accuracy')
    axes[0, 2].set_xticks(x)
    axes[0, 2].set_xticklabels(metrics)
    axes[0, 2].legend()
    axes[0, 2].grid(True, alpha=0.3)
    
    # Precision-Recall curve
    y_pred_aug = model_aug.predict(X_test)
    
    from sklearn.metrics import precision_recall_curve, auc
    precision, recall, _ = precision_recall_curve(y_test, y_pred_aug)
    pr_auc = auc(recall, precision)
    
    axes[1, 0].plot(recall, precision, color='blue', lw=2)
    axes[1, 0].fill_between(recall, precision, alpha=0.2, color='blue')
    axes[1, 0].set_title(f'Precision-Recall Curve\nAUC = {pr_auc:.3f}')
    axes[1, 0].set_xlabel('Recall')
    axes[1, 0].set_ylabel('Precision')
    axes[1, 0].grid(True, alpha=0.3)
    
    # Confusion Matrix
    from sklearn.metrics import confusion_matrix
    y_pred_binary = (y_pred_aug > 0.5).astype(int)
    cm = confusion_matrix(y_test, y_pred_binary)
    
    import seaborn as sns
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1, 1])
    axes[1, 1].set_title('Confusion Matrix (With Augmentation)')
    axes[1, 1].set_xlabel('Predicted')
    axes[1, 1].set_ylabel('Actual')
    
    # ROC Curve
    from sklearn.metrics import roc_curve, roc_auc_score
    fpr, tpr, _ = roc_curve(y_test, y_pred_aug)
    roc_auc = roc_auc_score(y_test, y_pred_aug)
    
    axes[1, 2].plot(fpr, tpr, color='darkorange', lw=2, label=f'AUC = {roc_auc:.3f}')
    axes[1, 2].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    axes[1, 2].set_title('ROC Curve')
    axes[1, 2].set_xlabel('False Positive Rate')
    axes[1, 2].set_ylabel('True Positive Rate')
    axes[1, 2].legend(loc='lower right')
    axes[1, 2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Fine-tuning demonstration
    print("\nDEMONSTRATING FINE-TUNING...")
    
    # Unfreeze some layers for fine-tuning
    vgg_base.trainable = True
    
    # Freeze first N layers, fine-tune the rest
    fine_tune_at = 15  # Fine-tune from this layer onwards
    
    for layer in vgg_base.layers[:fine_tune_at]:
        layer.trainable = False
    
    print(f"Fine-tuning {len(vgg_base.layers) - fine_tune_at} layers")
    
    # Recompile with lower learning rate
    model_aug.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-5),  # Lower learning rate
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    
    # Train for a few more epochs
    fine_tune_epochs = 10
    
    history_fine_tune = model_aug.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=fine_tune_epochs,
        batch_size=32,
        verbose=1
    )
    
    # Evaluate after fine-tuning
    test_loss_final, test_acc_final = model_aug.evaluate(X_test, y_test, verbose=0)
    print(f"\nAfter Fine-tuning - Test Accuracy: {test_acc_final:.4f}")
    print(f"Improvement: {test_acc_final - test_acc_aug:.4f}")
    
    # Compare different pre-trained models
    print("\nCOMPARING DIFFERENT PRE-TRAINED MODELS:")
    print("-" * 45)
    
    pre_trained_models = {
        'VGG16': keras.applications.VGG16,
        'ResNet50': keras.applications.ResNet50,
        'MobileNetV2': keras.applications.MobileNetV2,
        'EfficientNetB0': keras.applications.EfficientNetB0
    }
    
    model_results = {}
    
    for model_name, model_fn in pre_trained_models.items():
        print(f"\nBuilding {model_name}...")
        
        try:
            # Load pre-trained model
            base_model = model_fn(
                weights='imagenet',
                include_top=False,
                input_shape=(150, 150, 3)
            )
            base_model.trainable = False
            
            # Build transfer model
            inputs = keras.Input(shape=(150, 150, 3))
            
            # Preprocess based on model
            if 'vgg' in model_name.lower():
                x = keras.applications.vgg16.preprocess_input(inputs)
            elif 'resnet' in model_name.lower():
                x = keras.applications.resnet.preprocess_input(inputs)
            elif 'mobilenet' in model_name.lower():
                x = keras.applications.mobilenet_v2.preprocess_input(inputs)
            elif 'efficientnet' in model_name.lower():
                x = keras.applications.efficientnet.preprocess_input(inputs)
            else:
                x = inputs
            
            x = base_model(x, training=False)
            x = layers.GlobalAveragePooling2D()(x)
            x = layers.Dense(128, activation='relu')(x)
            x = layers.Dropout(0.5)(x)
            outputs = layers.Dense(1, activation='sigmoid')(x)
            
            model = keras.Model(inputs, outputs)
            
            model.compile(
                optimizer='adam',
                loss='binary_crossentropy',
                metrics=['accuracy']
            )
            
            # Train quickly
            history = model.fit(
                X_train, y_train,
                validation_split=0.2,
                epochs=5,
                batch_size=32,
                verbose=0
            )
            
            # Evaluate
            test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
            model_results[model_name] = {
                'test_accuracy': test_acc,
                'num_params': model.count_params(),
                'trainable_params': np.sum([np.prod(v.shape) for v in model.trainable_weights])
            }
            
            print(f"  Test Accuracy: {test_acc:.4f}")
            print(f"  Total Parameters: {model_results[model_name]['num_params']:,}")
            
        except Exception as e:
            print(f"  Error with {model_name}: {e}")
            model_results[model_name] = {'test_accuracy': 0, 'num_params': 0}
    
    # Visualize model comparison
    model_names = list(model_results.keys())
    accuracies = [model_results[name]['test_accuracy'] for name in model_names]
    params = [model_results[name]['num_params'] / 1e6 for name in model_names]  # In millions
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    
    # Accuracy comparison
    bars = ax1.bar(model_names, accuracies, color=['skyblue', 'lightcoral', 'lightgreen', 'gold'])
    ax1.set_title('Test Accuracy Comparison')
    ax1.set_ylabel('Accuracy')
    ax1.set_ylim(0, 1)
    ax1.tick_params(axis='x', rotation=45)
    
    for bar, acc in zip(bars, accuracies):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{acc:.3f}', ha='center', va='bottom')
    
    # Parameter comparison
    bars = ax2.bar(model_names, params, color=['skyblue', 'lightcoral', 'lightgreen', 'gold'])
    ax2.set_title('Model Size Comparison')
    ax2.set_ylabel('Number of Parameters (Millions)')
    ax2.tick_params(axis='x', rotation=45)
    
    for bar, param in zip(bars, params):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                f'{param:.1f}M', ha='center', va='bottom')
    
    plt.tight_layout()
    plt.show()
    
    return model_aug, model_results

# Jalankan latihan 2
transfer_model, model_comparison = praktikum_transfer_learning()


LATIHAN 2: TRANSFER LEARNING DENGAN DATA AUGMENTATION


ModuleNotFoundError: No module named 'tensorflow_datasets'